# A tour of the certified derived classification

This notebook classifies small gentle algebras over `F_2` up to derived
equivalence. Every merge is a replayable tilting recipe. Every separation
is a derived invariant whose two values differ. A pair that neither settles
is listed as unresolved.

Conventions: modules are right modules, and paths compose left to right.
In `Algebra(quiver, [[0, 1]])` the relation `[0, 1]` is arrow 0 followed by
arrow 1.

In [1]:
import auslander as au

F = au.PrimeField(2)
path_a3 = au.Algebra(au.Quiver(3, [(0, 1), (1, 2)]), [], field=F)
zero_a3 = au.Algebra(au.Quiver(3, [(0, 1), (1, 2)]), [[0, 1]], field=F)
print(au.presentation_text(zero_a3))

field 2
vertices 0 1 2
arrows a0: 0 -> 1 a1: 1 -> 2
relations a0*a1 = 0



## Invariants of two algebras

`derived_invariants()` computes each invariant in a fixed order. The path
algebra of `A_3` and its quotient by `a0*a1` agree on every finished value,
so `first_difference` returns `None`. Equal values never merge two
algebras.

In [2]:
left = path_a3.derived_invariants()
right = zero_a3.derived_invariants()
print(left)
print("first difference:", left.first_difference(right))

DerivedInvariants: 3 vertices, dim 6, field F_2, Hochschild degrees 0..2
  kind                   status    value
  vertex_count           finished  3
  cartan_determinant     finished  1
  cartan_factors         finished  [1, 1, 1]
  symmetric_factors      finished  [1, 1, 4]
  skew_factors           finished  [1, 1, 0]
  cartan_pencil          finished  x^3 + x^2 + x + 1
  aag_function           finished  [(4, 2)]
  winding_class          finished  planar
  hochschild_dimensions  finished  [1, 0, 0]
  center_dimension       finished  1
  a finished value that differs separates two algebras; equal values never merge them
first difference: None


## Gentle trees and `A_n`

Assem and Happel prove that a gentle algebra is derived equivalent to the
path algebra of `A_n` exactly when its quiver is a tree. The family below
has one algebra per isomorphism class of connected gentle bound quiver with
3 vertices. The class of the path algebra of `A_3` should be exactly the
trees, the members with 2 arrows.

In [3]:
family = au.connected_gentle_algebras(3, F)
result = au.classify_derived(family)
result

class,rep,members,merges,det C,AAG,winding,HH
0,0,"[0, 3, 12, 34]",3,0,"[(0, 1), (0, 2), (2, 1)]",planar,"[1, 3, 3]"
1,1,[1],0,0,"[(0, 1), (0, 2), (0, 2), (1, 0)]",planar,"[2, 4, 4]"
2,2,"[2, 4]",1,0,"[(0, 1), (0, 1), (0, 2), (1, 1)]",planar,"[1, 6, 5]"
3,5,"[5, 14, 15, 17, 27, 29]",5,4,"[(0, 1), (0, 1), (2, 2)]",planar,"[1, 4, 4]"
4,6,[6],0,8,"[(0, 1), (0, 1), (0, 1), (1, 2)]",planar,"[1, 6, 7]"
5,7,"[7, 8, 16, 26, 28, 30, 31]",6,2,"[(0, 1), (3, 2)]",planar,"[1, 2, 2]"
6,9,"[9, 13, 25, 33, 35]",4,2,"[(0, 1), (1, 1), (1, 2)]",planar,"[1, 4, 3]"
7,10,"[10, 22, 23, 24, 36]",4,2,"[(0, 1), (1, 4)]",gcd 1,"[1, 4, 2]"
8,11,"[11, 19, 21, 32]",3,2,"[(0, 1), (1, 0), (1, 3)]",planar,"[2, 3, 2]"
9,18,[18],0,8,"[(0, 1), (0, 1), (0, 3), (1, 0)]",planar,"[2, 5, 4]"


In [4]:
keys = au.connected_gentle_keys(3)
a3 = keys.index(path_a3.gentle().key)
trees = [index for index, member in enumerate(family) if member.num_arrows == 2]
a3_class = result.classes[result.class_of(a3)]
print("A_3 is member", a3)
print("trees:", trees)
print("class of A_3:", a3_class.members)
assert a3_class.members == trees

A_3 is member 63
trees: [37, 63, 64, 76]
class of A_3: [37, 63, 64, 76]


The same check at 4 vertices classifies only the 11 gentle trees, so it
proves one direction: every gentle tree with 4 vertices is derived
equivalent to `A_4`. It says nothing about the other members of the full
family.

In [5]:
trees4 = [member for member in au.connected_gentle_algebras(4, F) if member.num_arrows == 3]
tree_result = au.classify_derived(trees4)
print(len(trees4), "trees,", len(tree_result.classes), "class, status", tree_result.status)

11 trees, 1 class, status complete


## A genus-1 pair that only the winding class separates

Members 68 and 70 share the AAG function `[(2, 4)]`, the genus 1, and every
Cartan and Hochschild value. The winding class of Amiot, Plamondon, and
Schroll differs: `gcd 0` against `gcd 2`. The separation stores a witness
that recomputes both values.

In [6]:
for index in (68, 70):
    print(f"member {index}:", family[index].gentle().complete_invariant())
separation = next(s for s in result.separations if s.members == (68, 70))
witness = separation.witness
print(separation)
print(witness)
assert witness.verify()

member 68: GentleDerivedInvariant(aag_function=[(2, 4)], genus=1, winding_class="gcd 0")
member 70: GentleDerivedInvariant(aag_function=[(2, 4)], genus=1, winding_class="gcd 2")
ClassSeparation(classes=(24, 25), members=(68, 70), kind="winding_class")
DerivedInequivalenceWitness(kind="winding_class", left_value='gcd 0', right_value='gcd 2')


## A merge witness

A merge starts at the regular complex of one member and mutates it to a
tilting complex `T`. The recovered target `End(T)^op` maps onto a second
member through an algebra isomorphism. `show` writes the recipe, the vertex
map, and each arrow image in the arrows `a{i}` of the matched member.

In [7]:
merge = result.classes[result.class_of(44)].merges[0]
print(au.show(merge))
print()
print("member", merge.source)
print(au.presentation_text(family[merge.source]))
print("member", merge.member)
print(au.presentation_text(family[merge.member]))

DerivedMerge: member 44 to member 62
  recipe from the regular complex of member 44: left mutation at summand 1
  recovered target End(T)^op: dim 10, isomorphic to member 62
  vertex map: 0 -> 1, 1 -> 2, 2 -> 0
  arrow 0 -> a1
  arrow 1 -> a3
  arrow 2 -> a2
  arrow 3 -> a0

member 44
field 2
vertices 0 1 2
arrows a0: 0 -> 1 a1: 1 -> 0 a2: 0 -> 2 a3: 2 -> 1
relations a0*a1 = 0; a1*a2 = 0

member 62
field 2
vertices 0 1 2
arrows a0: 0 -> 1 a1: 1 -> 0 a2: 2 -> 0 a3: 2 -> 1
relations a0*a1 = 0; a2*a0 = 0



## Export, then verify

`export` writes the portable `derived-atlas-v1` atlas. `verify_derived_atlas`
rebuilds every member, recomputes every invariant, rebuilds every
separation witness, and replays every merge recipe. It runs no discovery.
The static page in `web/` replays the same file in a browser.
`to_latex` writes the class table for a paper.

In [8]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    path = result.export(Path(directory) / "gentle-f2-n3.json")
    replayed = au.verify_derived_atlas(path)
print(replayed.explain())
print("fingerprint:", replayed.fingerprint)
assert [c.members for c in replayed.classes] == [c.members for c in result.classes]
print("\n".join(au.to_latex(replayed).splitlines()[:9]))

DerivedClassification: status complete, verification replayed
  completed: 30 classes of 77 members; every pair of classes is separated by a derived invariant
fingerprint: 2277621c2c2270ef
% DerivedClassification: 77 members, 30 classes, status complete, verification replayed
% 47 merges by tilting recipes, 435 separations by invariants, 0 unresolved, 21 walks
% separations by invariant: cartan_determinant 332, cartan_factors 30, symmetric_factors 57, aag_function 15, winding_class 1
% derived-atlas-v1 fingerprint 2277621c2c2270ef
\begin{tabular}{rlrlll}
\hline
Class & Members & $\det C$ & AAG function & Winding class & $\dim \mathrm{HH}^i$, $i \le 2$ \\
\hline
0 & 0, 3, 12, 34 & 0 & $(0, 1), (0, 2), (2, 1)$ & planar & 1, 3, 3 \\
